# Softmax Regression — Gradient Ascent Algorithm

## Learning Objectives
- Express the multi-class cross-entropy loss as the **negative log-likelihood** over categorical predictions
- Derive the gradient $\nabla_\Theta J = -\frac{1}{m} X^T(Y - P)$ via the chain rule
- Understand the **log-sum-exp (max-shift) trick** for numerically stable softmax
- Implement batch gradient descent for softmax regression from scratch using only NumPy
- See how the gradient simplifies to the same **universal form** $X^T(\text{label} - \text{pred})$ shared with logistic regression

## Problem Statement

Given a training set $\{(x^{(i)}, y^{(i)})\}_{i=1}^{m}$ with $x^{(i)} \in \mathbb{R}^d$ and $y^{(i)} \in \{1, \ldots, K\}$, find a parameter matrix $\Theta \in \mathbb{R}^{(d+1) \times K}$ (one column per class) that minimises the average cross-entropy loss:

$\displaystyle \Theta^* = \arg\min_{\Theta}\; J(\Theta) = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} y_{ik} \log P(y^{(i)}=k \mid x^{(i)}; \Theta)$

where $Y \in \{0,1\}^{m \times K}$ is the one-hot encoding of labels.

---

### From Binary to Multi-Class

| Aspect | Logistic Regression ($K=2$) | Softmax Regression ($K > 2$) |
|--------|----|---------|
| Parameters | $\theta \in \mathbb{R}^{d+1}$ | $\Theta \in \mathbb{R}^{(d+1) \times K}$ |
| Output activation | Sigmoid $\sigma(\theta^T x)$ | Softmax $\frac{e^{\theta_k^T x}}{\sum_j e^{\theta_j^T x}}$ |
| Output shape | scalar $\in (0,1)$ | vector $\in \Delta^{K-1}$ (probability simplex) |
| Loss | Binary cross-entropy | Categorical cross-entropy |
| Gradient | $X^T(y - p)$ | $-\frac{1}{m} X^T(Y - P)$ |
| Identifiable? | Yes | No (add any constant to all columns) |

**Key insight**: Softmax regression has $K$ separate linear classifiers sharing the same features $x$, but their outputs are coupled through the normalisation denominator $\sum_j e^{\theta_j^T x}$. This coupling is what makes the gradient derivation interesting.

In [ ]:
from IPython.display import SVG, display

svg = """
<svg xmlns="http://www.w3.org/2000/svg" width="700" height="280" viewBox="0 0 700 280">
  <rect width="700" height="280" fill="#fafafa" rx="8"/>
  <text x="350" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#222">Softmax Regression — Gradient Descent Data Flow</text>

  <!-- Box 1: Input -->
  <rect x="15" y="50" width="100" height="60" rx="6" fill="#e8f4fd" stroke="#4a90d9" stroke-width="2"/>
  <text x="65" y="76" text-anchor="middle" font-size="10" font-weight="bold" fill="#4a90d9">Input X</text>
  <text x="65" y="94" text-anchor="middle" font-size="9" fill="#333">x &#x2208; &#x211D;&#x1D48;&#x207A;&#xB9;</text>
  <text x="65" y="108" text-anchor="middle" font-size="9" fill="#555">X: (m, d+1)</text>

  <!-- Arrow 1 -->
  <line x1="117" y1="80" x2="150" y2="80" stroke="#888" stroke-width="2"/>
  <polygon points="150,75 160,80 150,85" fill="#888"/>
  <text x="134" y="72" text-anchor="middle" font-size="8" fill="#555">X&#x398;</text>

  <!-- Box 2: Logits -->
  <rect x="162" y="50" width="110" height="60" rx="6" fill="#fff8e8" stroke="#e07b00" stroke-width="2"/>
  <text x="217" y="74" text-anchor="middle" font-size="10" font-weight="bold" fill="#e07b00">Logits Z</text>
  <text x="217" y="92" text-anchor="middle" font-size="9" fill="#333">Z = X&#x398; &#x2208; &#x211D;&#x1D48;&#xD7;&#x1D37;</text>
  <text x="217" y="108" text-anchor="middle" font-size="9" fill="#555">z&#x1D62;&#x2C7C; = &#x3B8;&#x2C7C;&#x1D40;x&#x207B;&#x2071;&#x207C;</text>

  <!-- Arrow 2 -->
  <line x1="274" y1="80" x2="307" y2="80" stroke="#888" stroke-width="2"/>
  <polygon points="307,75 317,80 307,85" fill="#888"/>
  <text x="291" y="72" text-anchor="middle" font-size="8" fill="#555">softmax</text>

  <!-- Box 3: Probabilities -->
  <rect x="319" y="50" width="120" height="60" rx="6" fill="#edfaed" stroke="#1a6e2e" stroke-width="2"/>
  <text x="379" y="71" text-anchor="middle" font-size="10" font-weight="bold" fill="#1a6e2e">Probs P</text>
  <text x="379" y="87" text-anchor="middle" font-size="9" fill="#333">P&#x1D62;&#x2C7C; = e^z&#x1D62;&#x2C7C; / &#x2211;e^z&#x1D62;&#x2113;</text>
  <text x="379" y="103" text-anchor="middle" font-size="9" fill="#333">P &#x2208; &#x211D;&#x1D48;&#xD7;&#x1D37;, rows sum=1</text>

  <!-- Arrow 3 -->
  <line x1="441" y1="80" x2="474" y2="80" stroke="#888" stroke-width="2"/>
  <polygon points="474,75 484,80 474,85" fill="#888"/>
  <text x="458" y="72" text-anchor="middle" font-size="8" fill="#555">cross-entropy</text>

  <!-- Box 4: Loss -->
  <rect x="486" y="50" width="120" height="60" rx="6" fill="#fde8e8" stroke="#c0392b" stroke-width="2"/>
  <text x="546" y="72" text-anchor="middle" font-size="10" font-weight="bold" fill="#c0392b">Loss J</text>
  <text x="546" y="90" text-anchor="middle" font-size="9" fill="#333">J = -(1/m)&#x2211;Y&#x2299;log P</text>
  <text x="546" y="106" text-anchor="middle" font-size="9" fill="#333">J &#x2208; &#x211D;, scalar</text>

  <!-- Backward arrows -->
  <text x="350" y="150" text-anchor="middle" font-size="11" font-weight="bold" fill="#333">&#x2190; Backpropagation (gradient descent update)</text>

  <!-- Gradient box -->
  <rect x="225" y="175" width="250" height="60" rx="6" fill="#f0e8fd" stroke="#7b2fc0" stroke-width="2"/>
  <text x="350" y="198" text-anchor="middle" font-size="11" font-weight="bold" fill="#7b2fc0">Gradient</text>
  <text x="350" y="216" text-anchor="middle" font-size="10" fill="#333">&#x2207;&#x398; J = -(1/m) X&#x1D40;(Y &#x2212; P) &#x2208; &#x211D;&#x1D48;&#x207A;&#xB9;&#xD7;&#x1D37;</text>
  <text x="350" y="230" text-anchor="middle" font-size="9" fill="#555">&#x398; &#x2190; &#x398; &#x2212; &#x3B1; &#x22C5; &#x2207;&#x398; J  (learning rate &#x3B1;)</text>

  <!-- Vertical arrows connecting Loss to Gradient -->
  <line x1="546" y1="112" x2="546" y2="165" stroke="#c0392b" stroke-width="1.5" stroke-dasharray="4,3"/>
  <line x1="546" y1="165" x2="475" y2="175" stroke="#c0392b" stroke-width="1.5" stroke-dasharray="4,3"/>

  <!-- Arrow from gradient to params -->
  <line x1="225" y1="205" x2="160" y2="205" stroke="#7b2fc0" stroke-width="1.5" stroke-dasharray="4,3"/>
  <line x1="160" y1="205" x2="100" y2="170" stroke="#7b2fc0" stroke-width="1.5" stroke-dasharray="4,3"/>
  <line x1="100" y1="170" x2="100" y2="112" stroke="#7b2fc0" stroke-width="1.5" stroke-dasharray="4,3"/>
  <polygon points="95,112 100,102 105,112" fill="#7b2fc0"/>
  <text x="50" y="165" text-anchor="middle" font-size="8" fill="#7b2fc0">update &#x398;</text>
</svg>
"""

display(SVG(svg))

## Model / Hypothesis

For input $x \in \mathbb{R}^{d+1}$ (bias prepended), the softmax regression model predicts:

$\displaystyle P(y = k \mid x; \Theta) = \frac{e^{\theta_k^T x}}{\sum_{j=1}^{K} e^{\theta_j^T x}}, \quad k = 1, \ldots, K$

where $\Theta = [\theta_1 \mid \cdots \mid \theta_K] \in \mathbb{R}^{(d+1) \times K}$ is the parameter matrix.

In matrix form for all $m$ samples:

$\displaystyle Z = X\Theta \in \mathbb{R}^{m \times K}, \qquad P_{ik} = \frac{e^{Z_{ik}}}{\sum_{j=1}^{K} e^{Z_{ij}}}$

---

**Cross-entropy loss** (negative log-likelihood under categorical model):

$\displaystyle J(\Theta) = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} Y_{ik} \log P_{ik} = -\frac{1}{m} \operatorname{tr}(Y^T \log P)$

where $Y \in \{0,1\}^{m \times K}$ is the one-hot label matrix.

**Numerical stability** — naive softmax $e^{z_{ik}} / \sum_j e^{z_{ij}}$ overflows for large logits. Fix with the **max-shift trick**:

$\displaystyle P_{ik} = \frac{e^{z_{ik} - c_i}}{\sum_j e^{z_{ij} - c_i}}, \qquad c_i = \max_j z_{ij}$

This is algebraically identical (the $e^{-c_i}$ factors cancel) but prevents overflow.

## Derivation

**Goal**: compute $\frac{\partial J}{\partial \theta_c}$ for each class $c$ via the chain rule.

---

**Step 1 — Log-likelihood for one sample**

$\displaystyle \log p(y^{(i)} \mid x^{(i)}) = \sum_{k=1}^{K} Y_{ik} \log P_{ik} = \log P_{i, y^{(i)}}$

---

**Step 2 — Gradient of the loss w.r.t. logit $z_{ic} = \theta_c^T x^{(i)}$**

Let $s_i = \sum_j e^{z_{ij}}$ (the softmax denominator). Then:

$\displaystyle \frac{\partial J}{\partial z_{ic}} = -\frac{1}{m}\left(Y_{ic} - \frac{e^{z_{ic}}}{s_i}\right) = -\frac{1}{m}(Y_{ic} - P_{ic})$

In matrix form: $\displaystyle \frac{\partial J}{\partial Z} = -\frac{1}{m}(Y - P) \in \mathbb{R}^{m \times K}$

---

**Step 3 — Gradient w.r.t. the parameter matrix $\Theta$**

Since $Z = X\Theta$, by the chain rule $\frac{\partial J}{\partial \Theta} = X^T \frac{\partial J}{\partial Z}$:

$\displaystyle \boxed{\nabla_\Theta J = -\frac{1}{m} X^T (Y - P) \in \mathbb{R}^{(d+1) \times K}}$

---

**Step 4 — Gradient descent update**

$\displaystyle \Theta^{(t+1)} = \Theta^{(t)} - \alpha \nabla_\Theta J = \Theta^{(t)} + \frac{\alpha}{m} X^T(Y - P)$

The update adds the weighted residuals $X^T(Y - P)$ — the same structural form as logistic and linear regression, but now $Y - P \in \mathbb{R}^{m \times K}$ is a residual matrix covering all classes.

## Algorithm: Batch Gradient Descent for Softmax Regression

**Step 1 — Initialise**

Set $\Theta \leftarrow \mathbf{0} \in \mathbb{R}^{(d+1) \times K}$. Prepend bias column to $X$. One-hot encode $y$ into $Y \in \{0,1\}^{m \times K}$.

**Step 2 — Compute logits**

$\displaystyle Z = X\Theta \in \mathbb{R}^{m \times K}$

**Step 3 — Stable softmax**

$\displaystyle Z' = Z - \max(Z, \text{axis}=1, \text{keepdims})$  (max-shift per row)

$\displaystyle P = e^{Z'} \,/\, \text{rowsum}(e^{Z'})$

**Step 4 — Gradient and update**

$\displaystyle \nabla_\Theta J = -\frac{1}{m} X^T(Y - P), \qquad \Theta \leftarrow \Theta - \alpha \nabla_\Theta J$

**Step 5 — Repeat** steps 2–4 until $\|\nabla_\Theta J\| < \varepsilon$ or maximum iterations reached.

| Quantity | Formula | Shape |
|----------|---------|-------|
| Logits | $Z = X\Theta$ | $(m, K)$ |
| Probabilities | $P_{ik} = e^{z_{ik}} / \sum_j e^{z_{ij}}$ | $(m, K)$ |
| One-hot labels | $Y_{ik} = \mathbf{1}[y^{(i)} = k]$ | $(m, K)$ |
| Residuals | $Y - P$ | $(m, K)$ |
| Gradient | $-\frac{1}{m} X^T(Y-P)$ | $(d+1, K)$ |
| Update | $\Theta \leftarrow \Theta - \alpha \nabla J$ | $(d+1, K)$ |

## Key Properties

**Universal gradient form** — the update $\Theta \leftarrow \Theta + \frac{\alpha}{m} X^T(Y-P)$ has the same structure as logistic regression gradient ascent. The only difference is that $Y$ and $P$ are now matrices ($m \times K$) rather than vectors.

**Non-identifiable** — adding the same constant to every column of $\Theta$ leaves $P$ unchanged (the constant cancels in softmax). The standard fix is to **tie** one class's parameter to zero ($\theta_K = 0$). In practice, L2 regularisation implicitly resolves the ambiguity.

**Reduction to logistic** — for $K = 2$, fixing $\theta_2 = 0$ gives $P(y=1 \mid x) = \sigma(\theta_1^T x)$, recovering logistic regression exactly. The cross-entropy loss and gradient also reduce to the binary case.

**Log-sum-exp trick** — numerically stable cross-entropy computation avoids computing $P$ explicitly:

$\displaystyle -\log P_{i,y^{(i)}} = -z_{i,y^{(i)}} + \log\sum_j e^{z_{ij}} = -z_{i,y^{(i)}} + c_i + \log\sum_j e^{z_{ij}-c_i}$

**Convergence** — cross-entropy loss is convex in $\Theta$ for fixed $K$, so gradient descent with a small enough $\alpha$ converges to the global minimum. Learning rate can be tuned; Newton's method (GLM approach) converges faster but costs more per step.

In [ ]:
import numpy as np


def softmax_stable(Z):
    """
    Numerically stable softmax using the max-shift (log-sum-exp) trick.

    Inputs
    ------
    Z : np.ndarray, shape (m, K)  — logit matrix (Z = X @ Theta)

    Output
    ------
    P : np.ndarray, shape (m, K)  — probability matrix; rows sum to 1
    """
    Z_shifted = Z - Z.max(axis=1, keepdims=True)   # subtract row-max for stability
    exp_Z     = np.exp(Z_shifted)
    return exp_Z / exp_Z.sum(axis=1, keepdims=True)


def one_hot(y, K):
    """
    Convert integer class labels to one-hot encoding.

    Inputs
    ------
    y : np.ndarray, shape (m,)  — integer labels in {0, ..., K-1}
    K : int                     — number of classes

    Output
    ------
    Y : np.ndarray, shape (m, K)  — one-hot matrix; Y[i, y[i]] = 1
    """
    Y = np.zeros((len(y), K))
    Y[np.arange(len(y)), y] = 1.0
    return Y


def fit(
    X, y, K,
    lr=0.1,
    max_iter=1000,
    tol=1e-6,
):
    """
    Fit softmax regression via batch gradient descent.

    Inputs
    ------
    X        : np.ndarray, shape (m, d+1)  — design matrix with bias column (x_0 = 1) prepended
    y        : np.ndarray, shape (m,)      — integer class labels in {0, ..., K-1}
    K        : int                         — number of classes
    lr       : float                       — learning rate alpha
    max_iter : int                         — maximum gradient descent iterations
    tol      : float                       — stop when ||gradient||_F < tol

    Outputs
    -------
    Theta   : np.ndarray, shape (d+1, K)  — learned parameter matrix (col k = theta_k)
    history : list of (int, float)         — (iteration, cross-entropy loss) at each step
    """
    m, d = X.shape
    Theta   = np.zeros((d, K))
    Y       = one_hot(y, K)            # (m, K) one-hot labels
    history = []

    for t in range(max_iter):
        Z    = X @ Theta               # (m, K)  logits
        P    = softmax_stable(Z)       # (m, K)  probabilities
        grad = -(1.0 / m) * (X.T @ (Y - P))   # (d+1, K)  gradient
        Theta = Theta - lr * grad      # gradient descent step

        # cross-entropy loss (log-sum-exp form for stability)
        lse  = np.log(np.exp(Z - Z.max(axis=1, keepdims=True)).sum(axis=1)) + Z.max(axis=1)
        loss = float(np.mean(lse - Z[np.arange(m), y]))
        history.append((t, loss))

        if np.linalg.norm(grad) < tol:
            break

    return Theta, history


def predict_proba(X, Theta):
    """
    Return class probability estimates.

    Inputs
    ------
    X     : np.ndarray, shape (m, d+1)    — design matrix with bias column (x_0 = 1) prepended
    Theta : np.ndarray, shape (d+1, K)   — learned parameter matrix

    Output
    ------
    P : np.ndarray, shape (m, K)  — P[i, k] = P(y=k | x^(i))
    """
    return softmax_stable(X @ Theta)


def predict(X, Theta):
    """
    Return predicted class labels (argmax over probabilities).

    Inputs
    ------
    X     : np.ndarray, shape (m, d+1)    — design matrix with bias column (x_0 = 1) prepended
    Theta : np.ndarray, shape (d+1, K)   — learned parameter matrix

    Output
    ------
    y_hat : np.ndarray, shape (m,)  — predicted integer class labels in {0, ..., K-1}
    """
    return predict_proba(X, Theta).argmax(axis=1)